# UPDATES NEEDED: 
This notebook creates Feature Views from multiple DeltaTableSources. It needs to be converted to use a combination of DeltaTableSource and StreamSource. If the below changes can be implemented successfully, the model should integrate with the real-time feature lookups seemlessly.



**TODO**:  

**1.** Create the PRODUCT_TABLE from 01_download_datasets. This table will be used for static features, like product name and categories. The Prod Batch Features cell below will use this table as its DeltaTableSource.  
**2.** Create an persist a new table based off the Stream's ingestion table, something like the below. It should contain all the clickstream features.

```# Expand the struct fields in 'value' into individual columns
flattened_df = ingestion.select(
    "value.event_time",
    "value.event_type",
    "value.product_id",
    "value.category_id",
    "value.category_code",
    "value.brand",
    "value.price",
    "value.user_id",
    "value.user_session",
    "record_source",
    "stream_record_timestamp",
)
```

This table will append click stream events as part of a batch job (does not need to be implemented) and used to derive product-level aggregates over a longer-time horizon, like 7-days. This effects the Product rolling aggregates cell.  

**3.** Convert the Features in the 'Session-Level Aggregate Features — Clickstream SlidingWindow' cell to use a **StreamSource** instead of the DeltaTableSource. This also requires changing the window type: `SlidingWindow(24h, 1h slide)` must become `RollingWindow(24h)`, because **RollingWindow is the only window type supported by StreamSource**. The existing aggregation operators (`Count`, `Avg`, `StddevPop`, `Sum`) are compatible with both window types and do not need to change.  

**4.** Confirm that all Features can be joined together using **create_training_set()**. 

**5.** Create a Feature Spec and serv it as a Feature Serving Endpoint; query the endpoint and ensure it returns all the features. NOTE: This would require streaming features through the StreamSource to populate the LakeBase table. One of those stream records would need to be passed into the serving request - only those sessions will exist in the Lakebase to be looked up.

#### NOTE: Do not remove or add any features at this point as their names are hard coded into the model where additional transformations are applied.


# Feature Store Integration — Two-Tower Recommender

Define Databricks Feature Views for real-time Feature calculations and online features lookups.

In [0]:
%pip install databricks-feature-engineering>=0.16.0
dbutils.library.restartPython()

%md
### Feature View Implementation

**TODO:** `session_category_diversity` is deferred. It requires `ApproxCountDistinct`, which is not supported on streaming `RollingWindow` features. Options to revisit:
1. Per-category `Count` features via streaming `RollingWindow` with `filter_condition` (14 features, one per L1 category)
2. Client-side tracking in the application layer
3. Ablation to determine if the feature carries enough signal to justify the complexity

#### Step 1: Split features into daily batch and real-time aggregate features.
Daily batch features are product-level features, including product ID, product category, and price. These features are updated as part of a batch job. Real-time aggregate features include two types of features, product-level aggegates and user session-level aggregates. Both of these features types are derived from sessions-level clickstream data, they exist in different online tables and their computation frequency is different.

Product-level aggregates are computed on session-level data, but they are update on a schedule cadence. Thinks of features like the number of purchases of a product over the past 7 days. These feature do not need to be calculated in near real time.

User session-level aggregates are capture in near real time via a StreamSource. These features are derived from a user's actions while browsing the website. They exist in their own Lakebase table. The StreamSource writes non-aggregated, raw clickstream events to a Delta table. This Delta table is used as a source to Derive product-level features.

In [0]:
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    DeltaTableSource, StreamSource, Feature, AggregationFunction,
    Count, LastDistinct, Max, Min, SlidingWindow, RollingWindow, Sum, Avg,
    StddevPop, RollingWindow,RequestSource, FieldDefinition, 
    ScalarDataType,  ColumnSelection, OfflineStoreConfig, 
    OnlineStoreConfig, CronSchedule, TableTrigger
)

from datetime import timedelta

fe = FeatureEngineeringClient()

In [0]:
"""
STREAM_CATALOG = config["catalog"]
STREAM_SCHEMA = config["schema"]
STREAM_NAME = config["stream_name"]
STREAM_SOURCE = f"{STREAM_CATALOG}.{STREAM_SCHEMA}.{STREAM_NAME}"
PRODUCT_AGGREGATE_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_aggregates_table']}"
"""

In [0]:
import yaml
import os
from databricks.feature_engineering import FeatureEngineeringClient

# Load project config
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
PRODUCT_TABLE = config['product_table']
PRODUCT_AGGREGATES_TABLE = config['product_aggregates_table']
STREAM_NAME = config["stream_name"]
STREAM_SOURCE = f"{CATALOG}.{SCHEMA}.{STREAM_NAME}"
ONLINE_STORE = config["online_store"]

# Create online store if it doesn't exist
fe = FeatureEngineeringClient()
store = fe.get_online_store(name=ONLINE_STORE)
if store is not None:
    print(f"Online store '{ONLINE_STORE}' exists (state: {store.state})")
else:
    fe.create_online_store(name=ONLINE_STORE, capacity="CU_1")
    print(f"Created online store '{ONLINE_STORE}'")

print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}\n")

print(f"Product table:   {PRODUCT_TABLE}")
print(f"Product aggregates table:   {PRODUCT_AGGREGATES_TABLE}")
print(f"Stream name:   {STREAM_NAME}\n")

print(f"Online store name:   {ONLINE_STORE}")

fe = FeatureEngineeringClient()

#### Create Feature Views

######Session-level rolling aggregate views

These features capture a user's browsing behavior within the current session.

In [0]:
import json

# --- Single StreamSource with transformation_sql ---
# Derives indicator columns for event-type counts and cart value,
# enabling all session features from one source (one materialization pipeline).
# Sessions with no events of a given type get 0 (not null) for counts and values.
PROJECTED_SCHEMA_JSON = json.dumps({
    "type": "struct",
    "fields": [
        {"name": "user_session", "type": "string", "nullable": True, "metadata": {}},
        {"name": "event_time", "type": "timestamp", "nullable": True, "metadata": {}},
        {"name": "event_type", "type": "string", "nullable": True, "metadata": {}},
        {"name": "product_id", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "brand", "type": "string", "nullable": True, "metadata": {}},
        {"name": "price", "type": "double", "nullable": True, "metadata": {}},
        {"name": "is_view", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_cart", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_purchase", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "cart_price", "type": "double", "nullable": True, "metadata": {}},
    ]
})

session_source = StreamSource(
    full_name=STREAM_SOURCE,
    transformation_sql=(
        "value.user_session AS user_session, "
        "value.event_time AS event_time, "
        "value.event_type AS event_type, "
        "value.product_id AS product_id, "
        "value.brand AS brand, "
        "value.price AS price, "
        "CASE WHEN value.event_type = 'view' THEN 1 ELSE 0 END AS is_view, "
        "CASE WHEN value.event_type = 'cart' THEN 1 ELSE 0 END AS is_cart, "
        "CASE WHEN value.event_type = 'purchase' THEN 1 ELSE 0 END AS is_purchase, "
        "CASE WHEN value.event_type = 'cart' THEN value.price ELSE 0.0 END AS cart_price"
    ),
    dataframe_schema=PROJECTED_SCHEMA_JSON,
)

SESSION_WINDOW = RollingWindow(
    window_duration=timedelta(hours=24),
)

# --- Session event count (browsing intensity) ---
session_event_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Count(input="event_type"), SESSION_WINDOW),
    name="session_event_count",
)

# --- Session average price (price tier browsing pattern) ---
session_avg_price = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Avg(input="price"), SESSION_WINDOW),
    name="session_avg_price",
)

# --- Session price standard deviation (price diversity) ---
session_price_stddev = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(StddevPop(input="price"), SESSION_WINDOW),
    name="session_price_stddev",
)

# --- Session total value (cumulative spend signal) ---
session_total_value = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="price"), SESSION_WINDOW),
    name="session_total_value",
)

# --- Session recently viewed products (browsing history) ---
session_recent_products = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(LastDistinct(input="product_id", n=10), SESSION_WINDOW),
    name="session_recent_products",
)

# --- Session view count (browsing breadth) ---
session_view_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_view"), SESSION_WINDOW),
    name="session_view_count",
)

# --- Session cart count (purchase intent) ---
session_cart_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_cart"), SESSION_WINDOW),
    name="session_cart_count",
)

# --- Session purchase count (conversion signal) ---
session_purchase_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_purchase"), SESSION_WINDOW),
    name="session_purchase_count",
)

# --- Session cart value (cumulative cart spend) ---
session_cart_value = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="cart_price"), SESSION_WINDOW),
    name="session_cart_value",
)

# --- Session recent brands (brand affinity) ---
session_recent_brands = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(LastDistinct(input="brand", n=5), SESSION_WINDOW),
    name="session_recent_brands",
)

# --- Collect all session rolling features ---
session_features = [
    session_event_count,
    session_avg_price,
    session_price_stddev,
    session_total_value,
    session_recent_products,
    session_view_count,
    session_cart_count,
    session_purchase_count,
    session_cart_value,
    session_recent_brands,
]

In [0]:
"""
# Verify via create_training_set — compute_features doesn't support StreamSource windowed features.
# The ingestion table backs the Stream; labeled df uses leaf column names (no value. prefix).
ingestion_df = spark.table(f"{STREAM_SOURCE}_ingestion")
labeled_df = ingestion_df.select(
    F.col("value.user_session").alias("user_session"),
    F.col("value.event_time").alias("event_time"),
    F.col("value.event_type").alias("label"),
).limit(1000)

training_set = fe.create_training_set(
    df=labeled_df,
    features=session_features,
    label="label",
)
session_df = training_set.load_df()
print(f"Session features computed: {session_df.count():,} rows")
display(session_df.limit(10))
"""

######Product batch views

Sourced from ecommerce_product (1 row/product/day). ColumnSelection resolves the latest attribute value per product BEFORE
each event's timestamp. The product catalog table handles attribute drift (price changes, recategorization, brand updates) with one row per product per day. In production, materialized to Lakebase via TableTrigger (refreshed whenever the source table changes).


In [0]:
product_source = DeltaTableSource(
    catalog_name=CATALOG,
    schema_name=SCHEMA,
    table_name=PRODUCT_TABLE,
)

# Static catalog features (latest value per product)
item_price = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("price"),
    name="item_price",
)

item_category_l1 = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("category_l1"),
    name="item_category_l1",
)

item_category_l2 = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("category_l2"),
    name="item_category_l2",
)

item_category_l3 = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("category_l3"),
    name="item_category_l3",
)

item_brand = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("brand"),
    name="item_brand",
)

item_log_price = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("log_price"),
    name="item_log_price",
)

product_batch_features = [
    item_price,
    item_category_l1,
    item_category_l2,
    item_category_l3,
    item_brand,
    item_log_price,
]

In [0]:
"""
product_batch_df = fe.compute_features(features=product_batch_features)
print(f"Product batch features computed: {product_batch_df.count():,} rows")
display(product_batch_df.limit(10))
"""

######Product rolling aggregate views
Product-level rolling aggregates — sourced from ecommerce_clickstream and ecommerce_product

These features capture item behavioral signals over a trailing 7-day SlidingWindow (slide=1d).
Updated on a daily CronSchedule in production (not real-time).

**Event counts** (from ecommerce_clickstream): A single DeltaTableSource with `transformation_sql` derives CASE WHEN indicator columns (`is_view`, `is_cart`, `is_purchase`) before aggregation, replacing multiple filtered sources. Event-type counts use `Sum` on indicators — products with no events of a given type get 0 (not null). `item_recency` uses `Max` on `event_time_copy` from the same source.

**Category price bounds** (from ecommerce_product): Min/Max price per category_l1 and category_l2, sourced from the product catalog (one row per product per day) for aggregation at product grain rather than event grain — avoids over-weighting popular items in the price range.

The pyfunc model derives final features at inference time:  

 - item_popularity = log1p(item_view_count). 
 - item_conversion_rate = (item_purchase_count + 1) / (item_view_count + 10)
 - item_cart_to_purchase_rate = (item_purchase_count + 1) / (item_cart_count + 10). 
 - recency_decay = 1.0 / (1.0 + days_since(item_recency, now)). 
 - price_pct_rank \~ (price - category_min) / (category_max - category_min)

In [0]:
# --- Single DeltaTableSource with transformation_sql ---
# Derives indicator columns for event-type counts, consolidating four sources into one.
# Products with no events of a given type get 0 (not null) for counts.
PRODUCT_AGG_SCHEMA_JSON = json.dumps({
    "type": "struct",
    "fields": [
        {"name": "product_id", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "event_time", "type": "timestamp", "nullable": True, "metadata": {}},
        {"name": "event_time_copy", "type": "timestamp", "nullable": True, "metadata": {}},
        {"name": "is_view", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_cart", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_purchase", "type": "integer", "nullable": True, "metadata": {}},
    ]
})

clickstream_source = DeltaTableSource(
    catalog_name=CATALOG,
    schema_name=SCHEMA,
    table_name=PRODUCT_AGGREGATES_TABLE,
    transformation_sql=(
        "product_id AS product_id, "
        "event_time AS event_time, "
        "event_time_copy AS event_time_copy, "
        "CASE WHEN event_type = 'view' THEN 1 ELSE 0 END AS is_view, "
        "CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END AS is_cart, "
        "CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END AS is_purchase"
    ),
    dataframe_schema=PRODUCT_AGG_SCHEMA_JSON,
)

PRODUCT_AGG_WINDOW = SlidingWindow(
    window_duration=timedelta(days=7),
    slide_duration=timedelta(days=1),
)

# --- Item event counts (mlflow pyfunc derives popularity, conversion rates)
item_view_count = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_view"), PRODUCT_AGG_WINDOW),
    name="item_view_count",
)

item_cart_count = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_cart"), PRODUCT_AGG_WINDOW),
    name="item_cart_count",
)

item_purchase_count = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_purchase"), PRODUCT_AGG_WINDOW),
    name="item_purchase_count",
)

# --- Item recency: most recent interaction timestamp per product
# Max(input="event_time") fails because event_time is the timeseries_column.
# Workaround: ecommerce_clickstream has event_time_copy (duplicate) which
# can be aggregated normally.
item_recency = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Max(input="event_time_copy"), PRODUCT_AGG_WINDOW),
    name="item_recency",
)

# --- Category-level price bounds (pyfunc derives price_pct_rank)
# percent_rank() is not available as a Feature Views operator.
# Approximation: store Min/Max price per category from the product catalog,
# then pyfunc computes price_pct_rank ≈ (price - min) / (max - min).
# Sourced from ecommerce_product (not clickstream) for cleaner aggregation
# over distinct product snapshots rather than event-weighted duplicates.
category_l1_min_price = Feature(
    source=product_source,
    entity=["category_l1"],
    timeseries_column="event_time",
    function=AggregationFunction(Min(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l1_min_price",
)

category_l1_max_price = Feature(
    source=product_source,
    entity=["category_l1"],
    timeseries_column="event_time",
    function=AggregationFunction(Max(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l1_max_price",
)

category_l2_min_price = Feature(
    source=product_source,
    entity=["category_l2"],
    timeseries_column="event_time",
    function=AggregationFunction(Min(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l2_min_price",
)

category_l2_max_price = Feature(
    source=product_source,
    entity=["category_l2"],
    timeseries_column="event_time",
    function=AggregationFunction(Max(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l2_max_price",
)

# --- Collect product rolling aggregate features ---
product_count_features = [
    item_view_count,
    item_cart_count,
    item_purchase_count,
    item_recency,
]

product_price_bound_features = [
    category_l1_min_price,
    category_l1_max_price,
    category_l2_min_price,
    category_l2_max_price,
]

product_rolling_features = product_count_features + product_price_bound_features

######Session-level rolling aggregate views

These features capture a user's browsing behavior within the current session.

Uses a single StreamSource with `transformation_sql` to derive indicator columns (`is_view`, `is_cart`, `is_purchase`, `cart_price`) from CASE WHEN expressions, enabling all session features from one source and one materialization pipeline. Event-type counts use `Sum` on indicators — sessions with no events of a given type get 0 (not null). Browsing history features use `LastDistinct` (works with `RollingWindow` despite not yet being in the public docs compatibility table).

`session_category_diversity` is DEFERRED — requires `ApproxCountDistinct`, which is not supported on `RollingWindow`. See cell 4 TODO for alternative approaches.

######RequestSource features
These are features that will ultimately be passed in to the model serving endpoint request; they are session-derived clickstream features passed at serving time.

These include the below types of features.
    - Properties of the current event (relevance, hour_of_day, day_of_week)
    - Entity keys needed for feature lookups at serving time (product_id, category_l1, category_l2)

In [0]:
request_source = RequestSource(
    schema=[
        FieldDefinition(name="product_id", data_type=ScalarDataType.INTEGER),
        FieldDefinition(name="category_l1", data_type=ScalarDataType.STRING),
        FieldDefinition(name="category_l2", data_type=ScalarDataType.STRING),
        FieldDefinition(name="relevance", data_type=ScalarDataType.INTEGER),
        FieldDefinition(name="hour_of_day", data_type=ScalarDataType.INTEGER),
        FieldDefinition(name="day_of_week", data_type=ScalarDataType.INTEGER),
    ]
)

product_id_feature = Feature(
    source=request_source,
    function=ColumnSelection("product_id"),
    name="product_id",
)

category_l1_feature = Feature(
    source=request_source,
    function=ColumnSelection("category_l1"),
    name="category_l1",
)

category_l2_feature = Feature(
    source=request_source,
    function=ColumnSelection("category_l2"),
    name="category_l2",
)

relevance_feature = Feature(
    source=request_source,
    function=ColumnSelection("relevance"),
    name="relevance",
)

hour_of_day_feature = Feature(
    source=request_source,
    function=ColumnSelection("hour_of_day"),
    name="hour_of_day",
)

day_of_week_feature = Feature(
    source=request_source,
    function=ColumnSelection("day_of_week"),
    name="day_of_week",
)

request_features = [
    product_id_feature,
    category_l1_feature,
    category_l2_feature,
    relevance_feature,
    hour_of_day_feature,
    day_of_week_feature,
]

print("RequestSource features (passed in serving payload):")
for f in request_features:
    print(f"  {f}")

In [0]:
# Test: Join all feature groups via create_training_set
# Cell 11: session_features (StreamSource + RollingWindow)
# Cell 14: product_batch_features (DeltaTableSource + ColumnSelection)
# Cell 17: product_rolling_features (DeltaTableSource + SlidingWindow)
# Cell 20: request_features (RequestSource)

all_test_features = (
    session_features
    + product_batch_features
    + product_rolling_features
    + request_features
)
print(f"Total features to join: {len(all_test_features)}")

# Build labeled df from ingestion table with all required entity/timeseries/request columns.
# StreamSource features use leaf names (no value. prefix) in the labeled df.
ingestion_df = spark.table(f"{STREAM_SOURCE}_ingestion")

labeled_df = (
    ingestion_df
    .select(
        F.col("value.product_id").alias("product_id"),
        F.col("value.user_session").alias("user_session"),
        F.col("value.event_time").alias("event_time"),
        F.col("value.event_type").alias("event_type"),
        F.col("value.category_code").alias("category_code"),
        F.col("value.price").alias("price"),
    )
    .limit(200)
    # Derive category_l1, category_l2 from category_code
    .withColumn("_parts", F.split(F.col("category_code"), "\\."))
    .withColumn("category_l1", F.coalesce(F.element_at("_parts", 1), F.lit("unknown")))
    .withColumn("category_l2", F.coalesce(F.element_at("_parts", 2), F.lit("unknown")))
    # Derive RequestSource columns
    .withColumn("relevance",
        F.when(F.col("event_type") == "purchase", 3)
         .when(F.col("event_type") == "cart", 2)
         .otherwise(1))
    .withColumn("hour_of_day", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_time") - 1)
    .withColumn("label", F.col("event_type"))
    .drop("_parts", "category_code", "event_type", "price")
)

training_set = fe.create_training_set(
    df=labeled_df,
    features=all_test_features,
    label="label",
)
combined_df = training_set.load_df()
print(f"Combined training set: {combined_df.count()} rows, {len(combined_df.columns)} columns")
print(f"Columns: {sorted(combined_df.columns)}")
display(combined_df.limit(10))

##### Register all features to Unity Catalog

Registration is required before materialization

In [0]:

all_registrable_features = (
    product_batch_features
    + product_rolling_features
    + session_features
    + request_features       
)

print(f"Registering {len(all_registrable_features)} features to {CATALOG}.{SCHEMA}...\n")

def register_features(features):
    """Register features to UC. If already registered, retrieve the existing object."""
    registered = []
    for f in features:
        try:
            r = fe.register_feature(feature=f, catalog_name=CATALOG, schema_name=SCHEMA)
            print(f"  ✓ {f.name}")
            registered.append(r)
        except Exception as e:
            if "already" in str(e).lower():
                r = fe.get_feature(full_name=f"{CATALOG}.{SCHEMA}.{f.name}")
                print(f"  • {f.name} (already registered)")
                registered.append(r)
            else:
                print(f"  ✗ {f.name}: {e}")
    return registered

In [0]:
product_batch_features = register_features(product_batch_features)
product_rolling_features = register_features(product_rolling_features)
session_features = register_features(session_features)
request_features = register_features(request_features)

all_registered = (product_batch_features + product_rolling_features
                  + session_features + request_features)
print(f"\nDone. {len(all_registered)} features registered to {CATALOG}.{SCHEMA}")

### Materialize Features

In [0]:
# ---------------------------------------------------------------------------
# Materialization schedules — driven by config.dev_mode
#   dev_mode=true  → every 5 minutes (fast iteration, data appears quickly)
#   dev_mode=false → production cadences (daily / hourly)
# ---------------------------------------------------------------------------

DEV_MODE = True

if DEV_MODE:
    PRODUCT_ROLLING_TRIGGER = CronSchedule(quartz_cron_expression="0 */5 * * * ?", timezone_id="UTC")
    SESSION_TRIGGER = CronSchedule(quartz_cron_expression="0 */5 * * * ?", timezone_id="UTC")
    print("⚡ Dev mode: materialization triggers set to every 5 minutes")
else:
    PRODUCT_ROLLING_TRIGGER = CronSchedule(quartz_cron_expression="0 0 0 * * ?", timezone_id="UTC")  # daily
    SESSION_TRIGGER = CronSchedule(quartz_cron_expression="0 0 * * * ?", timezone_id="UTC")           # hourly
    print("Production mode: product rolling = daily, session = hourly")

In [0]:
# ---------------------------------------------------------------------------
# Materialize product batch features (ColumnSelection) to Lakebase.
#
# ColumnSelection features use TableTrigger() — refreshed whenever the
# source table (ecommerce_product) changes. Online only (no offline store).
#
# Re-home these features into the configured online store by deleting any
# existing materializations first, then materializing again.
# ---------------------------------------------------------------------------

print("Re-materializing product batch features (6 ColumnSelection)...")
print(f"  Source: {CATALOG}.{SCHEMA}.{config['product_table']}")
print(f"  Trigger: TableTrigger (refresh on source change)")
print(f"  Target online store: {ONLINE_STORE}\n")

for f in product_batch_features:
    full_name = f"{CATALOG}.{SCHEMA}.{f.name}"
    mfs = fe.list_materialized_features(feature_name=full_name)
    for mf in mfs:
        fe.delete_materialized_feature(materialized_feature=mf)
        print(f"  Deleted existing materialization for {f.name}")

fe.materialize_features(
    features=product_batch_features,
    online_config=OnlineStoreConfig(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        table_name_prefix="product_batch",
        online_store_name=ONLINE_STORE,
    ),
    trigger=TableTrigger(),
)
print("\u2713 Product batch features materialized to Lakebase")

In [0]:
# ---------------------------------------------------------------------------
# Materialize product rolling aggregate features to Lakebase.
#
# AggregationFunction + SlidingWindow features use CronSchedule.
# AggregationFunction + CronSchedule requires both offline (Delta) and online
# (Lakebase) stores. The offline table is an intermediate computation layer
# that feeds Lakebase — do NOT use it as a training source (use
# create_training_set() instead, which computes point-in-time aggregations
# directly from the raw source with arbitrary train/eval splits).
#
# Re-home these features into the configured online store by deleting any
# existing materializations first, then materializing again.
# ---------------------------------------------------------------------------

schedule_desc = "every 5 min (dev)" if DEV_MODE else "daily at midnight UTC"
print(f"Re-materializing product rolling features (8 SlidingWindow aggregations)...")
print(f"  Sources: {config['ecommerce_table']} (counts) + {config['product_table']} (price bounds)")
print(f"  Trigger: CronSchedule ({schedule_desc})")
print(f"  Target online store: {ONLINE_STORE}\n")

for f in product_rolling_features:
    full_name = f"{CATALOG}.{SCHEMA}.{f.name}"
    mfs = fe.list_materialized_features(feature_name=full_name)
    offline = [mf for mf in mfs if not mf.is_online]
    for mf in (offline or mfs):
        fe.delete_materialized_feature(materialized_feature=mf)
        print(f"  Deleted existing materialization for {f.name}")

fe.materialize_features(
    features=product_rolling_features,
    offline_config=OfflineStoreConfig(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        table_name_prefix="product_rolling_offline",
    ),
    online_config=OnlineStoreConfig(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        table_name_prefix="product_rolling",
        online_store_name=ONLINE_STORE,
    ),
    trigger=PRODUCT_ROLLING_TRIGGER,
)
print("\u2713 Product rolling features materialized to Lakebase")

In [0]:
# ---------------------------------------------------------------------------
# Delete old SlidingWindow + CronSchedule materializations for session features.
# Required before re-materializing with StreamSource + StreamingMode.
# For aggregation features: delete offline (paired online is removed with it).
# ---------------------------------------------------------------------------

for f in session_features:
    full_name = f"{CATALOG}.{SCHEMA}.{f.name}"
    mfs = fe.list_materialized_features(feature_name=full_name)
    if not mfs:
        print(f"  {f.name}: no existing materializations")
        continue
    offline = [mf for mf in mfs if not mf.is_online]
    # Aggregation → delete offline (paired online goes with it).
    # If online-only (streaming), delete those directly.
    to_delete = offline or mfs
    for mf in to_delete:
        fe.delete_materialized_feature(materialized_feature=mf)
        print(f"  ✓ Deleted {'offline' if not mf.is_online else 'online'} materialization for {f.name}")

print("\nOld session materializations cleared — ready for StreamingMode.")

In [0]:
# ---------------------------------------------------------------------------
# Materialize session-level aggregate features to Lakebase.
#
# StreamSource + RollingWindow(24h) + StreamingMode (continuous).
# StreamSource features are online-only (no offline store).
# Operators: Count, Avg, StddevPop, Sum.
# ---------------------------------------------------------------------------

_session_mfs = fe.list_materialized_features(feature_name=f"{CATALOG}.{SCHEMA}.{session_features[0].name}")
if _session_mfs:
    print(f"Session features already materialized ({len(session_features)} features) \u2014 skipping")
else:
    print(f"Materializing session features (4 RollingWindow aggregations)...")
    print(f"  Source: StreamSource ({STREAM_SOURCE})")
    print(f"  Trigger: StreamingMode (continuous)")
    print(f"  Store: online (Lakebase) only\n")

    fe.materialize_features(
        features=session_features,
        online_config=OnlineStoreConfig(
            catalog_name=CATALOG,
            schema_name=SCHEMA,
            table_name_prefix="session_rolling",
            online_store_name=ONLINE_STORE,
        ),
        trigger=StreamingMode(),
    )
    print("\u2713 Session features materialized to Lakebase")

In [0]:
# ---------------------------------------------------------------------------
# Session features — migration COMPLETE
#
# Migrated from interim to production configuration:
#   SlidingWindow(24h, 1h) + DeltaTableSource + CronSchedule
#     → RollingWindow(24h) + StreamSource + StreamingMode
#
# Changes applied:
#   - Source: DeltaTableSource → StreamSource (clickstream_bf)
#   - Window: SlidingWindow → RollingWindow (only type supported by StreamSource)
#   - Trigger: CronSchedule → StreamingMode (continuous)
#   - Stores: offline + online → online only (Lakebase)
#   - Columns: bare names → value.-prefixed (Kafka payload struct)
#   - Operators (Count, Avg, StddevPop, Sum): unchanged
# ---------------------------------------------------------------------------

print("Session features — migration COMPLETE")
print("=" * 55)
print(f"\nFeatures: {len(session_features)}")
for f in session_features:
    print(f"  {f.name}")
print(f"\nConfiguration:")
print(f"  Source:  StreamSource ({STREAM_SOURCE})")
print(f"  Window:  RollingWindow(24h)")
print(f"  Trigger: StreamingMode (continuous)")
print(f"  Store:   online (Lakebase) only")

#### Deploy 'session_training_set' features to Feature Serving Endpoint
Only the session-level features need to be joined in real time.

In [0]:
from databricks.feature_engineering.entities.feature_serving_endpoint import (
    EndpointCoreConfig, ServedEntity,
)

# ---------------------------------------------------------------------------
# Retrieve all 25 registered features from UC (robust to session state)
# ---------------------------------------------------------------------------
FEATURE_NAMES = [
    # Session features (StreamSource + RollingWindow)
    "session_event_count", "session_avg_price", "session_price_stddev", "session_total_value",
    # Product batch features (ColumnSelection)
    "item_price", "item_category_l1", "item_category_l2", "item_category_l3", "item_brand", "item_log_price",
    # Product rolling features (SlidingWindow)
    "item_view_count", "item_cart_count", "item_purchase_count", "item_recency",
    "category_l1_min_price", "category_l1_max_price", "category_l2_min_price", "category_l2_max_price",
    # Request features (RequestSource)
    "product_id", "category_l1", "category_l2", "relevance", "session_cart_value", "hour_of_day", "day_of_week",
]

all_features = [fe.get_feature(full_name=f"{CATALOG}.{SCHEMA}.{name}") for name in FEATURE_NAMES]
print(f"Retrieved {len(all_features)} registered features from {CATALOG}.{SCHEMA}")

# ---------------------------------------------------------------------------
# Create FeatureSpec (idempotent)
# ---------------------------------------------------------------------------
SPEC_NAME = f"{CATALOG}.{SCHEMA}.two_tower_all_features_spec"
try:
    fe.create_feature_spec(name=SPEC_NAME, features=all_features)
    print(f"\u2713 Created FeatureSpec: {SPEC_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"\u2022 FeatureSpec already exists: {SPEC_NAME}")
    else:
        raise

# ---------------------------------------------------------------------------
# Recreate Feature Serving endpoint
# ---------------------------------------------------------------------------
ENDPOINT_NAME = "two-tower-feature-serving"
try:
    fe.delete_feature_serving_endpoint(name=ENDPOINT_NAME)
    print(f"\u2022 Deleted existing endpoint '{ENDPOINT_NAME}'")
except Exception:
    pass

fe.create_feature_serving_endpoint(
    name=ENDPOINT_NAME,
    config=EndpointCoreConfig(
        served_entities=ServedEntity(
            feature_spec_name=SPEC_NAME,
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ),
)
print(f"\u2713 Feature Serving endpoint '{ENDPOINT_NAME}' deployed")
print(f"  Spec: {SPEC_NAME}")
print(f"  Features: {len(all_features)}")
print(f"  Workload: Small, scale-to-zero enabled")

In [0]:
spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.session_features_online_veyalk").show()

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.session_features_online_veyalk"))

In [0]:
spark.sql(f"""
SELECT COUNT(DISTINCT user_session) AS distinct_sessions
FROM {CATALOG}.{SCHEMA}.session_features_online_veyalk
""").show()

In [0]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# -----------------------------------------------------------------------
# 1. Grab 10 sample sessions from the Lakebase session table
# -----------------------------------------------------------------------
samples = spark.sql(f"""
    SELECT user_session, event_time
    FROM {CATALOG}.{SCHEMA}.session_features_online_veyalk
    WHERE user_session IS NOT NULL
    LIMIT 10
""").collect()

print(f"Sampled {len(samples)} sessions from Lakebase\n")

# -----------------------------------------------------------------------
# 2. Get matching product_ids from the ingestion table for these sessions
# -----------------------------------------------------------------------
session_list = ",".join([f"'{r.user_session}'" for r in samples])
match_df = spark.sql(f"""
    SELECT value.user_session, value.product_id, value.event_time,
           value.category_code, value.event_type, value.price
    FROM {CATALOG}.{SCHEMA}.{config['stream_backfill_table']}
    WHERE value.user_session IN ({session_list})
""")

# Pick one event per session
from pyspark.sql import Window
w_spec = Window.partitionBy("user_session").orderBy(F.col("event_time").desc())
latest = (
    match_df
    .withColumn("rn", F.row_number().over(w_spec))
    .filter("rn = 1")
    .drop("rn")
    .collect()
)

print(f"Matched {len(latest)} sessions with product context\n")

# -----------------------------------------------------------------------
# 3. Build serving request records
# -----------------------------------------------------------------------
records = []
for row in latest:
    parts = (row.category_code or "unknown").split(".")
    cat_l1 = parts[0] if len(parts) >= 1 else "unknown"
    cat_l2 = parts[1] if len(parts) >= 2 else "unknown"
    relevance = 3 if row.event_type == "purchase" else (2 if row.event_type == "cart" else 1)
    records.append({
        "user_session": row.user_session,
        "product_id": row.product_id,
        "event_time": row.event_time.isoformat() if hasattr(row.event_time, 'isoformat') else str(row.event_time),
        "category_l1": cat_l1,
        "category_l2": cat_l2,
        "relevance": relevance,
        "session_cart_value": float(row.price or 0.0),
        "hour_of_day": row.event_time.hour if hasattr(row.event_time, 'hour') else 12,
        "day_of_week": row.event_time.weekday() if hasattr(row.event_time, 'weekday') else 0,
    })

print("Sample request record:")
import json
print(json.dumps(records[0], indent=2, default=str))

# -----------------------------------------------------------------------
# 4. Query the Feature Serving endpoint
# -----------------------------------------------------------------------
ENDPOINT_NAME = "two-tower-feature-serving"

print(f"\nQuerying endpoint '{ENDPOINT_NAME}' with {len(records)} records...\n")
resp = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=records,
)

import pandas as pd
result_df = pd.DataFrame(resp.outputs)
print(f"Returned {len(result_df)} rows, {len(result_df.columns)} columns")
print(f"Columns: {sorted(result_df.columns)}\n")
display(result_df)